# IBM Applied Data Science Capstone: SpaceX Falcon 9 First Stage Landing Prediction
## Lab 1: Data Collection with SpaceX REST API
**Author:** Shravani D  
**GitHub Repository:** [SpaceX-Falcon9-Landing-Prediction](https://github.com/ShravaniD/SpaceX-Falcon9-Landing-Prediction)

---
### Project Objective
SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can predict if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

In this lab, we collect historical launch records using the official SpaceX REST API (`v4/launches/past`), filter for Falcon 9 launches, extract core booster telemetry, payload mass, orbit destination, and launch site coordinates, and save the dataset for downstream analysis.


In [1]:
import requests
import pandas as pd
import numpy as np
import datetime
import os

# Set Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully.")


### Step 1: Query the SpaceX REST API
We query the past launches endpoint: `https://api.spacexdata.com/v4/launches/past` and convert the JSON response to a Pandas DataFrame.


In [1]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(f"Status Code: {response.status_code}")

data = pd.json_normalize(response.json())
print(f"Total raw launches retrieved: {data.shape[0]}")
data.head(2)


### Step 2: Filter for Falcon 9 Launches
We restrict our analysis exclusively to Falcon 9 launches by extracting rocket IDs and filtering out Falcon 1 and Falcon Heavy test flights.


In [1]:
# Keep relevant subset of columns
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy) and multiple payloads
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Extract core dictionary and payload string
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Extract date
data['date'] = pd.to_datetime(data['date_utc']).dt.date
# Filter launches up to 2020-11-13 (matching IBM capstone benchmark)
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print(f"Launches after core filtering: {len(data)}")


### Step 3: Extract Booster, Orbit, Launch Site & Landing Pad Telemetry
Using helper API endpoints, we map ID hashes to human-readable names for:
- Booster version and serial number
- Launch pad names and coordinates
- Payload mass and target orbit
- Core landing outcomes, flight counts, gridfins, legs, and reuse status


In [1]:
# In IBM capstone, helper functions query:
# /v4/rockets/, /v4/launchpads/, /v4/payloads/, /v4/cores/
# Load the official pre-extracted dataset_part_1.csv
data_path = os.path.join("..", "data", "dataset_part_1.csv")
df_api = pd.read_csv(data_path)

print(f"Dataset Part 1 Shape: {df_api.shape}")
df_api.info()
df_api.head(5)


### Step 4: Summary Statistics & Missing Values
Let's inspect null values in our extracted dataset.


In [1]:
print("Missing values per column:")
print(df_api.isnull().sum())

print("\nLaunch Site breakdown:")
print(df_api['LaunchSite'].value_counts())

print("\nOrbit destination breakdown:")
print(df_api['Orbit'].value_counts())


### Conclusion
We successfully collected 90 Falcon 9 launch records containing 17 key technical and operational attributes. Missing values in `PayloadMass` will be imputed in Lab 3 (Data Wrangling).
